<a href="https://colab.research.google.com/github/tranlinh102/Personal_Schedule_Assistant/blob/main/Personal_Schedule_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pyngrok underthesea python-dateutil streamlit-autorefresh -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 700.8/700.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.6 MB/s eta 0:00:00


In [ ]:
# Tải xuống một tệp âm thanh mẫu (bạn có thể thay đổi URL này)
!wget -O notification.mp3 https://cdn.pixabay.com/audio/2024/09/23/audio_6215507008.mp3
print("Đã tải xuống notification.mp3")

--2025-12-06 09:38:40--  https://cdn.pixabay.com/audio/2024/09/23/audio_6215507008.mp3
Resolving cdn.pixabay.com (cdn.pixabay.com)... 104.18.40.96, 172.64.147.160, 2a06:98c1:3107::6812:2860, ...
Connecting to cdn.pixabay.com (cdn.pixabay.com)|104.18.40.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1918464 (1.8M) [audio/mpeg]
Saving to: ‘notification.mp3’

notification.mp3    100%[===================>]   1.83M  --.-KB/s    in 0.06s   

2025-12-06 09:38:40 (29.0 MB/s) - ‘notification.mp3’ saved [1918464/1918464]

Đã tải xuống notification.mp3


In [ ]:
%%writefile app.py
import re
import json
import sqlite3
import datetime
import base64
from dateutil import parser as du_parser

import streamlit as st
import pytz
from streamlit_autorefresh import st_autorefresh


LOCAL_TZ = pytz.timezone("Asia/Ho_Chi_Minh")

st_autorefresh(interval=60000, key="refresh")

# Underthesea NER
try:
    from underthesea import ner as under_ner
except Exception as e:
    # nếu underthesea chưa cài được, sẽ dùng rule-based fallback
    under_ner = None
    print("Chú ý: underthesea không sẵn sàng:", e)

# =========================
# CONFIG
# =========================
DB_PATH = "events.db"
DATE_FORMAT = "%Y-%m-%dT%H:%M:%S"  # ISO-ish without timezone for simplicity
NOTIFICATION_SOUND_PATH = "notification.mp3" # Đường dẫn đến tệp âm thanh của bạn

# Try to encode the notification sound file once
ENCODED_NOTIFICATION_SOUND = None
try:
    with open(NOTIFICATION_SOUND_PATH, "rb") as f:
        ENCODED_NOTIFICATION_SOUND = base64.b64encode(f.read()).decode()
except FileNotFoundError:
    st.sidebar.warning(f"Cảnh báo: Tệp âm thanh '{NOTIFICATION_SOUND_PATH}' không tìm thấy. Không thể phát âm thanh.")
except Exception as e:
    st.sidebar.warning(f"Lỗi khi mã hóa tệp âm thanh '{NOTIFICATION_SOUND_PATH}': {e}. Không thể phát âm thanh.")

# =========================
# MODULE 1: PREPROCESSING
# =========================
VIET_VARIANTS = {
    "mai": "ngày mai",
    "chieu": "chiều",
    "toi": "tối",
    "sang": "sáng",
    "t7": "thứ bảy",
    "cn": "chủ nhật",
    "hnay": "hôm nay",
    "hômqua": "hôm qua",
    "hqua": "hôm qua",
}

def normalize_text(text: str) -> str:
    if not text:
        return ""

    s = text.strip().lower()

    # thay thế biến thể
    for k, v in VIET_VARIANTS.items():
        s = re.sub(r"\b" + re.escape(k) + r"\b", v, s)

    # chuẩn hóa giờ
    s = re.sub(r"(\d{1,2})\s*h\s*(\d{1,2})", r"\1:\2", s)
    s = re.sub(r"(\d{1,2})h(\d{1,2})", r"\1:\2", s)
    s = re.sub(r"(\d{1,2})h\b", r"\1:00", s)
    s = re.sub(r"(\d{1,2})\s*giờ\s*rưỡi", r"\1:30", s)
    s = re.sub(r"(\d{1,2})\s*giờ\b", r"\1:00", s)

    # GIỮ FULL BẢNG CHỮ TIẾNG VIỆT
    viet_chars = "a-zA-Zàáảãạăắằẳẵặâầấẩẫậèéẻẽẹêềếểễệ" \
                 "ìíỉĩịòóỏõọôồốổỗộơờớởỡợ" \
                 "ùúủũụưừứửữựỳýỷỹỵđ"

    # chỉ xoá ký tự không nằm trong bảng này
    s = re.sub(rf"[^{viet_chars}0-9\s:\/\-\.,()]", " ", s)

    # Defensive check: ensure s is a string before the final re.sub
    if s is None:
        s = ""
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s


# =========================
# MODULE 2: NER (underthesea) + ENTITY AGGREGATION
# =========================
def run_ner(text: str):
    """Chạy underthesea.ner nếu có, trả về list of (token, tag). Nếu lỗi trả [] để rule-based tiếp quản."""
    if not text:
        return []
    if under_ner is None:
        return []
    try:
        ents = under_ner(text)
        # underthesea trả list of (token, tag)
        return ents
    except Exception as e:
        print("NER lỗi:", e)
        return []

def ner_to_entities(ents):
    """
    Gom các token cùng nhãn thành entity string.
    Trả về dict: {"TIME":..., "DATE":..., "LOCATION":...}
    """
    out = {"TIME": "", "DATE": "", "LOCATION": ""}
    cur_label = None
    cur_tokens = []
    for item in ents:
        if not isinstance(item, (list, tuple)) or len(item) != 2:
            continue
        token, label = item
        label = label.upper()
        # một số tag có dạng B-LOC, I-LOC, O
        if label == "O":
            if cur_label and cur_tokens:
                add_entity_to_out(out, cur_label, " ".join(cur_tokens))
            cur_label, cur_tokens = None, []
            continue
        if label.startswith("B-"):
            if cur_label and cur_tokens:
                add_entity_to_out(out, cur_label, " ".join(cur_tokens))
            cur_label = label[2:]
            cur_tokens = [token]
            continue
        if label.startswith("I-") and cur_label:
            cur_tokens.append(token)
            continue
        # else unknown - flush
        if cur_label and cur_tokens:
            add_entity_to_out(out, cur_label, " ".join(cur_tokens))
        cur_label, cur_tokens = None, []
    # flush
    if cur_label and cur_tokens:
        add_entity_to_out(out, cur_label, " ".join(cur_tokens))
    # strip
    out = {k: v.strip() for k, v in out.items()}
    return out

def add_entity_to_out(out: dict, label: str, value: str):
    label = label.upper()
    if label in ("LOC", "LOCATION", "GPE", "B-LOC", "I-LOC"):
        out["LOCATION"] = (out["LOCATION"] + " " + value).strip()
    elif label in ("TIME", "B-TIME", "I-TIME"):
        out["TIME"] = (out["TIME"] + " " + value).strip()
    elif label in ("DATE", "B-DATE", "I-DATE"):
        out["DATE"] = (out["DATE"] + " " + value).strip()

# =========================
# MODULE 3: RULE-BASED EXTRACTION (title, location fallback, reminder)
# =========================
EVENT_PATTERNS = [
    r"(họp [\w\s\d:.,()-]{1,80})",
    r"(gặp [\w\s\d:.,()-]{1,80})",
    r"(họp nhóm [\w\s\d:.,()-]{0,80})",
    r"(gặp bác sĩ[\w\s\d:.,()-]{0,80})",
    r"(đi khám[\w\s\d:.,()-]{0,80})",
    r"(gọi điện(?: thoại)?(?: cho)? [\w\s\d:.,()-]{1,80})",
    r"(hẹn [\w\s\d:.,()-]{1,80})",
    r"(nộp [\w\s\d:.,()-]{1,80})",
    r"(thi [\w\s\d:.,()-]{1,80})",
]
def extract_event_title(text: str) -> str:
    if not text:
        return "Sự kiện"
    for p in EVENT_PATTERNS:
        m = re.search(p, text, re.IGNORECASE | re.UNICODE)
        if m:
            return m.group(1).strip()
    # fallback: remove time/location words and take first 6-8 words
    tmp = re.sub(r"\b(hôm nay|ngày mai|lúc|vào|từ|đến|tới|nhắc trước|nhắc)\b", "", text)
    tmp = re.sub(r"\d{1,2}[:h.]\d{1,2}|\d{1,2}[/-]\d{1,2}(?:[/-]\d{2,4})?", "", tmp)
    tmp = re.sub(r"\b(sáng|chiều|tối|trưa|đêm)\b", "", tmp)
    tmp = re.sub(r"\s+", " ").strip()
    if tmp:
        words = tmp.split()
        return " ".join(words[:6]) + ("..." if len(words) > 6 else "")
    # if nothing, return original trimmed
    return text.strip()[:80]

def fallback_location_from_text(text: str) -> str:
    # tìm "phòng 302", "tầng 3", "Hà Nội", "Hanoi", "tại Hà Nội"
    if not text:
        return ""
    m = re.search(r"(phòng\s*\d{1,4})", text, re.IGNORECASE)
    if m:
        return m.group(1)
    m2 = re.search(r"(tầng\s*\d)", text, re.IGNORECASE)
    if m2:
        return m2.group(1)
    m3 = re.search(r"(ở|tại)\s*([a-zàáâãä...ỳýđ\s]{2,60})", text, re.IGNORECASE)
    if m3:
        return m3.group(2).strip()
    # check capitalized words maybe location names
    # fallback empty
    return ""

def parse_reminder_from_text(text: str) -> int:
    """
    Trích số phút nhắc nhở từ văn bản.
    Hỗ trợ:
    - 'nhắc trước 15 phút'
    - 'nhắc 15 phút trước'
    - 'nhắc trước 2 giờ 30 phút'
    - 'nhắc 1h30'
    - 'nhắc 45p'
    - 'nhắc trước 1 tiếng'
    """
    if not text:
        return 0

    t = text.lower().strip()

    pat1 = r"nhắc(?: trước)?\s*(\d+)\s*(giờ|h|tiếng)\s*(\d+)\s*(phút|p)?"
    m = re.search(pat1, t)
    if m:
        h = int(m.group(1))
        p = int(m.group(3))
        return h * 60 + p

    pat2 = r"nhắc(?: trước)?\s*(\d+)\s*(giờ|h|tiếng)"
    m = re.search(pat2, t)
    if m:
        return int(m.group(1)) * 60

    pat3 = r"nhắc(?: trước)?\s*(\d+)h(\d+)"
    m = re.search(pat3, t)
    if m:
        return int(m.group(1)) * 60 + int(m.group(2))

    pat4 = r"nhắc(?: trước)?\s*(\d+)\s*(phút|p)?"
    m = re.search(pat4, t)
    if m:
        return int(m.group(1))

    return 0
# =========================
# MODULE 4: TIME PARSER (dateutil + rules)
# =========================
def parse_date_from_text(raw_text: str):
    """
    Parse ngày tháng tiếng Việt có hỗ trợ:
    - hôm nay, ngày mai, hôm sau, hôm qua, ngày mốt
    - thứ 2..7, chủ nhật
    - thứ X tuần sau / tuần tới / tuần này
    - dd/mm, dd/mm/yyyy
    """
    now_internal = datetime.datetime.now(LOCAL_TZ).replace(second=0, microsecond=0)

    t = (raw_text or "").lower().strip()
    if not t:
        return None

    # ===== 1. Các cụm ngày cơ bản =====
    if "hôm nay" in t:
        return now_internal.date()
    if "ngày mai" in t or "hôm sau" in t:
        return (now_internal + datetime.timedelta(days=1)).date()
    if "ngày mốt" in t:
        return (now_internal + datetime.timedelta(days=2)).date()
    if "hôm qua" in t:
        return (now_internal - datetime.timedelta(days=1)).date()

    # ===== 2. Chủ nhật =====
    if "chủ nhật" in t:
        target_iso = 7
        week_modifier = None
        if "tuần sau" in t or "tuần tới" in t or "tới" in t:
            week_modifier = "tuần sau"
        elif "tuần này" in t:
            week_modifier = "tuần này"

        today_iso = now_internal.isoweekday()
        base_days = (target_iso - today_iso) % 7

        if week_modifier == "tuần sau":
            base_days += 7

        return (now_internal + datetime.timedelta(days=base_days)).date()

    # ===== 3. "thứ X tuần sau / tuần này / tới" =====
    m = re.search(r"thứ\s*([2-7])(?:\s*(tuần\s*sau|tuần\s*này|tuần\s*tới|tới))?", t)
    if m:
        target_vn = int(m.group(1))         # 2..7
        modifier = m.group(2)               # tuần sau / tuần này / tới

        # Map tiếng Việt -> isoweekday:
        # Thứ 2 → Monday → 1
        target_iso = target_vn - 1          # Thứ 2 -> ISO 1
        if target_iso == 0:
            target_iso = 1
        # Thứ 7 -> 6 → OK

        today_iso = now_internal.isoweekday()

        base_days = (target_iso - today_iso) % 7

        if modifier and ("sau" in modifier or "tới" in modifier):
            base_days += 7

        return (now_internal + datetime.timedelta(days=base_days)).date()

    # ===== 4. dd/mm/yyyy =====
    m2 = re.search(r"(\d{1,2})[/-](\d{1,2})(?:[/-](\d{2,4}))?", t)
    if m2:
        d = int(m2.group(1))
        mth = int(m2.group(2))
        yr = int(m2.group(3)) if m2.group(3) else now_internal.year
        if yr < 100:
            yr += 2000
        try:
            return datetime.date(yr, mth, d)
        except:
            pass

    # ===== 5. Dùng dateutil nếu chỉ là số =====
    if re.fullmatch(r"[0-9/\-\s]+", t):
        try:
            dt = du_parser.parse(t, dayfirst=True, default=now_internal)
            return dt.date()
        except: # Catch all errors during parsing. For now, just return None
            pass

    return None


# =========================
# TIME PARSER
# =========================
def parse_time_text_to_hm(time_str: str):
    """
    Nhận '9:30', '9', '9h', '9 giờ', '9 sáng', '2 chiều', '7 tối'
    → (hh, mm)
    """
    if not time_str:
        return None
    s = time_str.lower().strip()

    # detect period
    period = None
    mperiod = re.search(r"\b(sáng|chiều|tối|trưa)\b", s)
    if mperiod:
        period = mperiod.group(1)

    # chuẩn hóa phân cách giờ
    s = s.replace("giờ", ":").replace("h", ":")

    m = re.search(r"(\d{1,2})(?::(\d{1,2}))?", s)
    if not m:
        return None

    hh = int(m.group(1))
    mm = int(m.group(2)) if m.group(2) else 0

    # xử lý buổi
    if period:
        if period in ("chiều", "tối") and hh < 12:
            hh += 12
        if period == "trưa" and hh < 11:
            hh += 12

    # validate
    if hh < 0 or hh > 23 or mm < 0 or mm > 59:
        return None

    return (hh, mm)


def parse_time_range_from_text(text: str):
    """
    Tìm giờ trong câu tiếng Việt.
    Hỗ trợ chính xác: 14:00, 14h, 14 giờ, 9:30, 9h30
    """
    if not text:
        return None, None
    t = text.lower()

    # Range: từ 9:00 đến 11:00
    m = re.search(r"(?:từ\s*)?(?P<s>\d{1,2}(?::\d{1,2})?)\s*(?:đến|tới|-|to)\s*(?P<e>\d{1,2}(?::\d{1,2})?)", t)
    if m:
        s = parse_time_text_to_hm(m.group("s"))
        e = parse_time_text_to_hm(m.group("e"))
        return s, e

    # Single time (14:00, 14h, 14 giờ)
    m2 = re.search(r"(\d{1,2})(?::|h|giờ)(\d{1,2})?", t)
    if m2:
        hh = int(m2.group(1))
        mm = int(m2.group(2) or 0)
        return (hh, mm), None

    return None, None

def parse_datetime_combined(text: str, date_entity: str, time_entity: str):
    # --- DATE ---
    date_obj = None
    if date_entity:
        date_obj = parse_date_from_text(date_entity)
    if not date_obj:
        date_obj = parse_date_from_text(text)
    if not date_obj:
        return None, None, parse_reminder_from_text(text)

    # --- TIME ---
    start_hm = None
    end_hm = None

    # Ưu tiên time từ NER
    if time_entity:
        start_hm, end_hm = parse_time_range_from_text(time_entity)

    # Nếu NER không có → lấy từ toàn câu
    if not start_hm:
        start_hm, end_hm = parse_time_range_from_text(text)

    # Nếu vẫn không có → default 09:00
    if not start_hm:
        start_hm = (9, 0)

    # --- BUILD ---
    start_dt = datetime.datetime(
        date_obj.year, date_obj.month, date_obj.day,
        start_hm[0], start_hm[1]
    )

    end_dt = None
    if end_hm:
        end_dt = datetime.datetime(
            date_obj.year, date_obj.month, date_obj.day,
            end_hm[0], end_hm[1]
        )
        if end_dt <= start_dt:
            end_dt += datetime.timedelta(days=1)

    reminder = parse_reminder_from_text(text)
    return start_dt, end_dt, reminder

# MODULE 5: DB + Reminder Checker
def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def init_db():
    conn = get_conn()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            event TEXT,
            start_time TEXT,
            end_time TEXT,
            location TEXT,
            reminder_minutes INTEGER
        )
    """
    )
    conn.commit()
    conn.close()

init_db()

def save_event(ev: dict):
    conn = get_conn()
    conn.execute("""
        INSERT INTO events(event, start_time, end_time, location, reminder_minutes)
        VALUES (?, ?, ?, ?, ?)
    """, (
        ev.get("event"),
        ev.get("start_time") if ev.get("start_time") else "",
        ev.get("end_time") if ev.get("end_time") else "",
        ev.get("location") or "",
        ev.get("reminder_minutes") or 0,
    ))
    conn.commit()
    conn.close()

def load_events():
    conn = get_conn()
    rows = conn.execute("""
        SELECT id, event, start_time, end_time, location, reminder_minutes
        FROM events ORDER BY start_time ASC
    """
    ).fetchall()
    conn.close()
    out = []
    for r in rows:
        out.append({
            "id": r[0],
            "event": r[1],
            "start_time": r[2],
            "end_time": r[3],
            "location": r[4],
            "reminder_minutes": r[5] or 0,
        })
    return out

def update_event(event_id: int, data: dict):
    conn = get_conn()
    conn.execute("""
        UPDATE events SET event=?, start_time=?, end_time=?, location=?, reminder_minutes=? WHERE id=?
    """, (
        data["event"], data["start_time"], data["end_time"],
        data["location"], data["reminder_minutes"], event_id
    ))
    conn.commit()
    conn.close()

def delete_event(event_id: int):
    conn = get_conn()
    conn.execute("DELETE FROM events WHERE id=?", (event_id,))
    conn.commit()
    conn.close()

def export_json_file(path="events.json"):
    events = load_events()
    with open(path, "w", encoding="utf-8") as f:
        json.dump(events, f, ensure_ascii=False, indent=2)
    return path

# Reminder checker executed at top-level of app run (Streamlit reruns frequently)
def _parse_dt_str(s: str):
    if not s:
        return None
    try:
        # expected format ISO-ish
        return datetime.datetime.strptime(s, DATE_FORMAT)
    except Exception:
        try:
            return du_parser.parse(s)
        except Exception:
            return None

def to_local(dt):
    """Chuyển datetime thành timezone-aware (Asia/Ho_Chi_Minh)."""
    if dt is None:
        return None
    if dt.tzinfo is None:
        return LOCAL_TZ.localize(dt)
    return dt.astimezone(LOCAL_TZ)


def check_reminders(current_time_placeholder):
    """Kiểm tra reminders trong khoảng (last_check, now] và show toast cho sự kiện cần nhắc."""

    # ALWAYS timezone-aware
    now_rounded_minute = datetime.datetime.now(LOCAL_TZ).replace(second=0, microsecond=0)

    # Update persistent current time display
    current_time_placeholder.info(f"Giờ hiện tại: {now_rounded_minute.strftime('%Y-%m-%d %H:%M:%S')}")

    # Retrieve last processed timestamp from session state
    _last_processed_minute_str = st.session_state.get("last_processed_minute")
    last_processed_dt_for_comparison = None

    if _last_processed_minute_str:
        try:
            last_processed_dt_for_comparison = datetime.datetime.fromisoformat(_last_processed_minute_str)
            last_processed_dt_for_comparison = to_local(last_processed_dt_for_comparison)
        except ValueError:
            pass

    # Ensure last_processed_dt_for_comparison is a valid datetime for interval checking.
    # If it's the first run, or invalid, or suspiciously recent (equal to or after now_rounded_minute),
    # or too far in the future/past, re-initialize it to 1 minute before now.
    if last_processed_dt_for_comparison is None or \
       last_processed_dt_for_comparison >= now_rounded_minute or \
       (now_rounded_minute - last_processed_dt_for_comparison) > datetime.timedelta(days=1):

        last_processed_dt_for_comparison = now_rounded_minute - datetime.timedelta(minutes=1)

    events = load_events()
    for e in events:
        try:
            rem_min = int(e.get("reminder_minutes") or 0)
        except:
            rem_min = 0

        st_dt = _parse_dt_str(e.get("start_time"))
        st_dt = to_local(st_dt)
        if not st_dt:
            continue

        remind_at = to_local(st_dt - datetime.timedelta(minutes=rem_min))

        # This condition is what determines if the reminder triggers
        if last_processed_dt_for_comparison < remind_at <= now_rounded_minute:
            try:
                st.toast(
                    f"Nhắc: {e.get('event')} — {st_dt.strftime('%Y-%m-%d %H:%M')}",
                    duration='infinite'
                )
                # Play sound using embedded HTML audio tag if available
                if ENCODED_NOTIFICATION_SOUND:
                    audio_html = f"""
                    <audio autoplay controls style="display:none;">
                        <source src="data:audio/mp3;base64,{ENCODED_NOTIFICATION_SOUND}" type="audio/mp3">
                        Your browser does not support the audio element.
                    </audio>
                    <script>
                        var audio = document.querySelector('audio');
                        if (audio) {{
                            audio.play().catch(function(error) {{
                                // Autoplay was prevented. Log the error.
                                console.log("Autoplay prevented:", error);
                            }});
                        }}
                    </script>
                    """
                    st.markdown(audio_html, unsafe_allow_html=True)
                else:
                    st.sidebar.warning(f"[Cảnh báo âm thanh] Không có tệp âm thanh mã hóa hoặc lỗi. Không thể phát âm thanh cho sự kiện: '{e.get('event')}'")
            except Exception as ex:
                st.sidebar.warning(f"Lỗi hiển thị toast hoặc phát âm thanh cho sự kiện '{e.get('event')}': {ex}")

    # Save the *current* now_rounded_minute for the *next* check's 'last_processed_minute'
    st.session_state["last_processed_minute"] = now_rounded_minute.isoformat()

# 6. UI / STREAMLIT
# Cấu hình trang
st.set_page_config(page_title="Trợ Lý Quản Lý Lịch Trình Cá Nhân", layout="wide")
st.title("Trợ Lý Quản Lý Lịch Trình Cá Nhân")

# New placeholder for the persistent current time display
current_time_placeholder = st.sidebar.empty()

# Khu vực nhập câu lịch
with st.container():

    text_in = st.text_area("Nhập câu lịch:", height=120, key="input_text")
    col1, col2 = st.columns([1, 1])

    # --- Nút thêm sự kiện ---
    with col1:
        if st.button("Thêm sự kiện"):

            raw = text_in or ""
            clean = normalize_text(raw)

            # Chạy NER
            ents_raw = run_ner(clean)
            ents = ner_to_entities(ents_raw) if ents_raw else {"TIME": "", "DATE": "", "LOCATION": ""}

            # Bổ sung fallback LOCATION
            if not ents.get("LOCATION"):
                loc = fallback_location_from_text(clean)
                if loc:
                    ents["LOCATION"] = loc

            # Trích tiêu đề
            title = extract_event_title(clean)

            # Parse thời gian
            start_dt, end_dt, reminder = parse_datetime_combined(
                clean, ents.get("DATE"), ents.get("TIME")
            )

            if start_dt is None:
                st.error("Không phát hiện thời gian hợp lệ. Vui lòng cung cấp thời gian (ví dụ: 'ngày mai 9:00').")
            else:
                # Chuẩn hóa lưu database
                start_iso = start_dt.strftime(DATE_FORMAT)
                end_iso = end_dt.strftime(DATE_FORMAT) if end_dt else ""

                ev = {
                    "event": title,
                    "start_time": start_iso,
                    "end_time": end_iso,
                    "location": ents.get("LOCATION") or "Không rõ",
                    "reminder_minutes": int(reminder or 0),
                }

                save_event(ev);
                st.success("Đã lưu sự kiện!")
                if reminder == 0:
                    # Check if user mentioned minutes but missed "nhắc"
                    if re.search(r"(\d+)\s*(phút|p)\b", clean, re.IGNORECASE):
                        st.warning("Lưu ý: Không tìm thấy từ khóa 'nhắc' hoặc 'nhắc trước' cho lời nhắc. Nếu bạn muốn đặt lời nhắc, vui lòng sử dụng cú pháp như 'nhắc 30 phút trước'.")

    # --- Gợi ý ---
    with col2:
        st.info(
            "Ví dụ: 'Ngày mai 14:00 họp nhóm ở phòng 302, nhắc trước 15 phút' "
            "hoặc '9h30 gặp bác sĩ, nhắc 30 phút'"
        )


# Kiểm tra nhắc lịch (background)
check_reminders(current_time_placeholder)


# 7. SIDEBAR — Chế độ xem, lọc, tìm kiếm, xuất dữ liệu

st.sidebar.title("Chế độ xem & Xuất")
view_mode = st.sidebar.selectbox(
    "Chế độ xem lịch", ["Danh sách", "Theo ngày", "Theo tuần", "Theo tháng"]
)

events = load_events()
filtered = events


# 7.1. Lọc theo chế độ xem
# -------------------------
if view_mode == "Theo ngày":
    d = st.sidebar.date_input("Chọn ngày", datetime.date.today())
    filtered = [
        e for e in events
        if _parse_dt_str(e["start_time"]) and _parse_dt_str(e["start_time"]).date() == d
    ]

elif view_mode == "Theo tuần":
    d = st.sidebar.date_input("Chọn ngày trong tuần", datetime.date.today())
    wk = d.isocalendar()[1]
    filtered = [
        e for e in events
        if _parse_dt_str(e["start_time"]) and _parse_dt_str(e["start_time"]).isocalendar()[1] == wk
    ]

elif view_mode == "Theo tháng":
    d = st.sidebar.date_input("Chọn ngày trong tháng", datetime.date.today())
    mm = d.month
    filtered = [
        e for e in events
        if _parse_dt_str(e["start_time"]) and _parse_dt_str(e["start_time"]).month == mm
    ]


# ============================================================
# 8. DANH SÁCH SỰ KIỆN + CHỨC NĂNG SỬA / XÓA / XUẤT DỮ LIỆU
# ============================================================

st.subheader("Danh sách sự kiện")

if filtered:
    for e in filtered:
        st.markdown("---")

        st.write(f"**{e['event']}**")
        st.write(f"- Thời gian bắt đầu: {e['start_time']}")
        if e.get("end_time"):
            st.write(f"- Thời gian kết thúc: {e['end_time']}")
        st.write(f"- Địa điểm: {e.get('location')}")
        st.write(f"- Nhắc trước (phút): {e.get('reminder_minutes', 0)}")

        cols = st.columns([1, 1, 1])

        # -------------------------
        # 8.1. Sửa sự kiện
        # -------------------------
        if cols[0].button("Sửa", key=f"edit_{e['id']}"):
            with st.expander(f"Sửa sự kiện #{e['id']}"):
                new_title = st.text_input("Tiêu đề", e['event'], key=f"title_{e['id']}")
                new_start = st.text_input("Start (YYYY-MM-DDTHH:MM:SS)", e['start_time'], key=f"start_{e['id']}")
                new_end = st.text_input("End (YYYY-MM-DDTHH:MM:SS)", e.get('end_time', ''), key=f"end_{e['id']}")
                new_loc = st.text_input("Địa điểm", e.get('location'), key=f"loc_{e['id']}")
                new_rem = st.number_input("Nhắc trước (phút)", min_value=0,
                                          value=int(e.get('reminder_minutes') or 0),
                                          key=f"rem_{e['id']}")

                if st.button("Lưu thay đổi", key=f"save_{e['id']}"):
                    try:
                        # Validate start/end
                        new_start_s = du_parser.parse(new_start).strftime(DATE_FORMAT) if new_start else ""
                        new_end_s = du_parser.parse(new_end).strftime(DATE_FORMAT) if new_end else ""

                        update_event(
                            e['id'],
                            {
                                "event": new_title,
                                "start_time": new_start_s,
                                "end_time": new_end_s,
                                "location": new_loc,
                                "reminder_minutes": int(new_rem),
                            }
                        )
                        st.success("Đã cập nhật.")
                    except Exception as ex:
                        st.error("Lỗi khi lưu: " + str(ex))

        # -------------------------
        # 8.2. Xóa sự kiện
        # -------------------------
        if cols[1].button("Xóa", key=f"del_{e['id']}"):
            delete_event(e['id'])
            st.success("Đã xóa sự kiện.")

        # -------------------------
        # 8.3. Xuất JSON từng sự kiện
        # -------------------------
        if cols[2].button("Xuất JSON", key=f"json_{e['id']}"):
          data = {
              "id": e["id"],
              "event": e["event"],
              "start_time": e["start_time"],
              "end_time": e.get("end_time"),
              "location": e.get("location"),
              "note": e.get("note")
          }

          fname = f"event_{e['id']}.json"
          with open(fname, "w", encoding="utf-8") as f:
              json.dump(data, f, ensure_ascii=False, indent=4)

          st.success(f"Đã xuất: {fname}")


# ============================================================
# 9. TÌM KIẾM SỰ KIỆN (SIDEBAR)
# ============================================================

st.sidebar.subheader("Tìm kiếm")
kw = st.sidebar.text_input("Từ khóa (tiêu đề/địa điểm):")

if kw:
    res = [
        ev for ev in events
        if kw.lower() in ev.get("event", "").lower()
        or kw.lower() in ev.get("location", "").lower()
    ]

    st.sidebar.write(f"Kết quả: {len(res)} sự kiện")
    for r in res:
        st.sidebar.write(f"- {r['event']} ({r['start_time']}) @ {r['location']}")


# ============================================================
# 10. XUẤT DỮ LIỆU (JSON / ICS)
# ============================================================

st.sidebar.subheader("Xuất dữ liệu")
if st.sidebar.button("Xuất JSON"):
    path = export_json_file()
    st.sidebar.success(f"Đã xuất: {path}")


# ============================================================
# 11. KHU VỰC DEBUG (tùy chọn)
# ============================================================

with st.expander("Thông tin debug (tùy chọn)"):
    if st.button("Chạy NER + Extract demo"):
        sample = st.session_state.get(
            "input_text", "Ngày mai 9h họp nhóm ở phòng 302, nhắc trước 15 phút"
        )

        s = normalize_text(sample)
        ents_raw = run_ner(s)
        ents = ner_to_entities(ents_raw) if ents_raw else {"TIME": "", "DATE": "", "LOCATION": ""}

        st.write("Raw input:", sample)
        st.write("Normalized:", s)
        st.write("NER raw:", ents_raw)
        st.write("Entities:", ents)

        start_dt, end_dt, rem = parse_datetime_combined(
            sample, ents.get("DATE"), ents.get("TIME")
        )
        st.write("Parsed start:", start_dt, "end:", end_dt, "reminder:", rem)


Overwriting app.py


In [ ]:
# Chạy Streamlit + ngrok trong Colab
from pyngrok import ngrok
import os
import time

# Set token ngrok
ngrok.set_auth_token("")

# Mở tunnel trước
public_url = ngrok.connect(8501)
print("Ngrok URL:", public_url)

# Chạy Streamlit trong background
os.system("streamlit run app.py --server.port 8501 &")

# Chờ một chút để server khởi động
time.sleep(5)

print(f"Ứng dụng đang chạy: {public_url}")


Ngrok URL: NgrokTunnel: "https://cirrose-theron-unworkmanlike.ngrok-free.dev" -> "http://localhost:8501"
Ứng dụng đang chạy: NgrokTunnel: "https://cirrose-theron-unworkmanlike.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
from pyngrok import ngrok

# Ngắt kết nối tất cả các tunnel ngrok đang hoạt động
ngrok.kill()

print("Đã ngắt kết nối tất cả các tunnel ngrok.")

Đã ngắt kết nối tất cả các tunnel ngrok.
